Devo nomear as colunas igual no exmplo do kaggle?


In [1]:
import pandas as pd
import numpy as np
import networkx as nx
from scipy.stats import zscore, skew, kurtosis

# ==============================
# 1. CARREGAMENTO
# ==============================

features = pd.read_csv("datasets/elliptic_txs_features.csv", header=None)
edges = pd.read_csv("datasets/elliptic_txs_edgelist.csv")
classes = pd.read_csv("datasets/elliptic_txs_classes.csv")

features.rename(columns={0: "txId"}, inplace=True)

df = features.merge(classes, on="txId", how="left")

# Mantém unknown
df["class"] = df["class"].astype(str)

# ==============================
# 2. ESTATÍSTICAS GERAIS
# ==============================

print("\n===== ESTATÍSTICAS GERAIS =====")
print("Número total de nós:", df.shape[0])
print("Número de features:", df.shape[1] - 2)

print("\nDistribuição de classes (%):")
print(df["class"].value_counts(normalize=True) * 100)

# ==============================
# 3. OUTLIERS (somente features)
# ==============================

numeric_cols = df.drop(columns=["txId", "class"]).columns

# Z-score
z_scores = np.abs(zscore(df[numeric_cols], nan_policy='omit'))
outliers_z = (z_scores > 3).sum().sum()
total_values = df[numeric_cols].shape[0] * df[numeric_cols].shape[1]
outlier_z_pct = (outliers_z / total_values) * 100

# IQR
Q1 = df[numeric_cols].quantile(0.25)
Q3 = df[numeric_cols].quantile(0.75)
IQR = Q3 - Q1

outliers_iqr = ((df[numeric_cols] < (Q1 - 1.5 * IQR)) |
                (df[numeric_cols] > (Q3 + 1.5 * IQR))).sum().sum()

outlier_iqr_pct = (outliers_iqr / total_values) * 100

print(f"\nOutliers Z-score (%): {outlier_z_pct:.2f}")
print(f"Outliers IQR (%): {outlier_iqr_pct:.2f}")

# ==============================
# 4. CONSTRUÇÃO DO GRAFO
# ==============================

G = nx.from_pandas_edgelist(edges, source="txId1", target="txId2")

print("\n===== MÉTRICAS DE GRAFO =====")
print("Número de nós no grafo:", G.number_of_nodes())
print("Número de arestas:", G.number_of_edges())
print("Densidade:", nx.density(G))
print("Componentes conectados:", nx.number_connected_components(G))

degrees = dict(G.degree())
degree_values = list(degrees.values())

print("Grau médio:", np.mean(degree_values))
print("Grau máximo:", np.max(degree_values))
print("Desvio padrão do grau:", np.std(degree_values))

# ==============================
# 5. CENTRALIDADES
# ==============================

print("\nCalculando centralidades...")

degree_centrality = nx.degree_centrality(G)
betweenness = nx.betweenness_centrality(G, k=500)
closeness = nx.closeness_centrality(G)
pagerank = nx.pagerank(G)

centralities = pd.DataFrame({
    "txId": list(degree_centrality.keys()),
    "degree_centrality": list(degree_centrality.values()),
    "betweenness": list(betweenness.values()),
    "closeness": list(closeness.values()),
    "pagerank": list(pagerank.values())
})

df = df.merge(centralities, on="txId", how="left")

# ==============================
# 6. MÉDIAS POR CLASSE (incluindo unknown)
# ==============================

print("\n===== CENTRALIDADES MÉDIAS POR CLASSE =====")
print(df.groupby("class")[[
    "degree_centrality",
    "betweenness",
    "closeness",
    "pagerank"
]].mean())

# ==============================
# 7. ESTATÍSTICAS DAS FEATURES
# ==============================

stats = pd.DataFrame({
    "mean": df[numeric_cols].mean(),
    "std": df[numeric_cols].std(),
    "skew": df[numeric_cols].apply(skew),
    "kurtosis": df[numeric_cols].apply(kurtosis)
})

stats.to_csv("elliptic_feature_statistics.csv")

print("\nAnálise concluída e exportada.")


===== ESTATÍSTICAS GERAIS =====
Número total de nós: 203769
Número de features: 166

Distribuição de classes (%):
class
unknown    77.148634
2          20.620899
1           2.230467
Name: proportion, dtype: float64

Outliers Z-score (%): 1.29
Outliers IQR (%): 13.41

===== MÉTRICAS DE GRAFO =====
Número de nós no grafo: 203769
Número de arestas: 234355
Densidade: 1.1288341056918834e-05
Componentes conectados: 49
Grau médio: 2.3002026804862368
Grau máximo: 473
Desvio padrão do grau: 4.328366155605154

Calculando centralidades...

===== CENTRALIDADES MÉDIAS POR CLASSE =====
         degree_centrality   betweenness  closeness  pagerank
class                                                        
1                 0.000010  7.954218e-07   0.001820  0.000005
2                 0.000015  2.022109e-06   0.002415  0.000006
unknown           0.000010  1.372760e-06   0.002057  0.000005

Análise concluída e exportada.


In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ks_2samp
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import LabelEncoder

# ==============================
# 1. CARREGAMENTO
# ==============================

features = pd.read_csv("datasets/elliptic_txs_features.csv", header=None)
classes = pd.read_csv("datasets/elliptic_txs_classes.csv")
edges = pd.read_csv("datasets/elliptic_txs_edgelist.csv")

features = features.rename(columns={0: "txId"})
df = features.merge(classes, on="txId", how="left")

# Mantém unknown
df["class"] = df["class"].astype(str)

numeric_cols = df.drop(columns=["txId", "class"]).columns

# ==============================
# 2. CONSTRUÇÃO DO GRAFO
# ==============================

G = nx.from_pandas_edgelist(
    edges, source="txId1", target="txId2", create_using=nx.DiGraph()
)

# adicionar atributo class aos nós (incluindo unknown)
class_dict = df.set_index("txId")["class"].to_dict()
nx.set_node_attributes(G, class_dict, "class")

# ==============================
# 3. ASSORTATIVIDADE
# ==============================

assortativity = nx.attribute_assortativity_coefficient(G, "class")
print("\nAssortatividade por classe:", assortativity)

# ==============================
# 4. CLUSTERING COEFFICIENT POR CLASSE
# ==============================

clustering = nx.clustering(G.to_undirected())
df["clustering"] = df["txId"].map(clustering)

print("\nClustering médio por classe:")
print(df.groupby("class")["clustering"].mean())

# ==============================
# 5. DISTRIBUIÇÃO DE GRAU POR CLASSE
# ==============================

degrees = dict(G.degree())
df["degree"] = df["txId"].map(degrees)

plt.figure()
sns.histplot(data=df, x="degree", hue="class", bins=50, kde=True)
plt.title("Distribuição de Grau por Classe")
plt.savefig("elliptic_degree_distribution.png")
plt.close()

# ==============================
# 6. CORRELAÇÃO (Pearson + Spearman)
# ==============================

pearson_corr = df[numeric_cols].corr(method="pearson")
spearman_corr = df[numeric_cols].corr(method="spearman")

pearson_corr.to_csv("elliptic_pearson_corr.csv")
spearman_corr.to_csv("elliptic_spearman_corr.csv")

# ==============================
# 7. KOLMOGOROV-SMIRNOV (3 comparações)
# ==============================

ks_results = []

classes_unique = df["class"].unique()

for col in numeric_cols:
    for i in range(len(classes_unique)):
        for j in range(i+1, len(classes_unique)):
            c1 = classes_unique[i]
            c2 = classes_unique[j]

            group1 = df[df["class"] == c1][col]
            group2 = df[df["class"] == c2][col]

            stat, p = ks_2samp(group1, group2)
            ks_results.append((col, c1, c2, stat, p))

ks_df = pd.DataFrame(
    ks_results,
    columns=["feature", "class_1", "class_2", "ks_stat", "p_value"]
)

ks_df.sort_values("ks_stat", ascending=False).to_csv(
    "elliptic_ks_test_all_classes.csv", index=False
)

# ==============================
# 8. MUTUAL INFORMATION (multiclasse)
# ==============================

le = LabelEncoder()
y_encoded = le.fit_transform(df["class"])

mi = mutual_info_classif(df[numeric_cols], y_encoded)

mi_df = pd.DataFrame({"feature": numeric_cols, "mi": mi})
top_features = mi_df.sort_values("mi", ascending=False)["feature"].head(5)

for col in top_features:
    plt.figure()
    sns.boxplot(x="class", y=col, data=df)
    plt.title(f"Boxplot - {col}")
    plt.savefig(f"elliptic_boxplot_{col}.png")
    plt.close()

print("\nEDA estrutural Elliptic concluída.")